# PA Traffic Observations — Matching rows only

Loads the hourly `pa_traffic_observations.log-YYYYMMDDHH` files from `H:/Observation2.0/pa_traffic_observations/`, parses only the *Matching rows* section in each file, and builds **`matches_df`**: one row per session that hit an observation.

Each row is tagged with `source_hour` (from the filename) and `source_file` so you can group, filter, and pivot easily. Duplicate-count tables in the same files are ignored.

After cleanup, rows are left-joined to **`Splunk_TC_Address_Index.csv`** on `matched_ip` = `indicator` to produce **`matches_tc_df`**, then **sorted by `matched_ip`** (string order on the normalized address).

In [13]:
import os
import re
import glob
from io import StringIO
from datetime import datetime

import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
pd.set_option('display.max_colwidth', 60)

LOG_DIR = r'H:/Observation2.0/pa_traffic_observations'
files = sorted(glob.glob(os.path.join(LOG_DIR, 'pa_traffic_observations.log-*')))
print(f'Found {len(files)} files')
print('First:', os.path.basename(files[0]))
print('Last: ', os.path.basename(files[-1]))

Found 190 files
First: pa_traffic_observations.log-2026050510
Last:  pa_traffic_observations.log-2026051307


## Parser

Each file has a `Duplicate counts:` section followed by `Matching rows:`. We locate `Matching rows:` and parse only that fixed-width table, then tag rows with `source_hour` (from the filename suffix) and `source_file`.

In [14]:
HOUR_RE = re.compile(r'log-(\d{10})$')


def hour_from_filename(path):
    m = HOUR_RE.search(os.path.basename(path))
    return datetime.strptime(m.group(1), '%Y%m%d%H') if m else None


def parse_file(path):
    with open(path, 'r', encoding='utf-8', errors='replace') as f:
        text = f.read()

    if 'Matching rows:' in text:
        _, match_part = text.split('Matching rows:', 1)
    else:
        match_part = ''

    match_df = pd.DataFrame()
    match_lines = [ln for ln in match_part.splitlines() if ln.strip()]
    if match_lines:
        buf = StringIO('\n'.join(match_lines))
        try:
            match_df = pd.read_fwf(buf)
            if str(match_df.columns[0]).lower().startswith('unnamed'):
                match_df = match_df.drop(columns=match_df.columns[0])
        except Exception as e:
            print(f'  ! could not parse matching rows in {os.path.basename(path)}: {e}')

    return match_df


_m = parse_file(files[0])
print('Sample file:', os.path.basename(files[0]))
print('  matching rows:', len(_m))
_m.head()

Sample file: pa_traffic_observations.log-2026050510
  matching rows: 48


,Unnamed: 1,timestamp,session_id,port_number,application,action,matching_ip,matched_column,matched_ip,total_bytes,sent_bytes,received_bytes,total_packets,packets_sent,packets_received,reason_for_session_end
0,2026/05/05,09:13:29,6191117,47759,incomplete,allow,193.163.125.51,src_ip,193.163.125.51,62,62,0,1,1,0,aged-out
1,2026/05/05,09:13:29,1149322950,41415,ssdp,allow,172.202.118.67,src_ip,172.202.118.67,141,141,0,1,1,0,aged-out
2,2026/05/05,09:13:29,1074243713,33108,incomplete,allow,193.163.125.52,src_ip,193.163.125.52,60,60,0,1,1,0,aged-out
3,2026/05/05,09:13:28,1110873964,37310,incomplete,allow,193.163.125.31,src_ip,193.163.125.31,60,60,0,1,1,0,aged-out
4,2026/05/05,09:13:28,1111004683,48793,incomplete,allow,91.191.209.46,src_ip,91.191.209.46,60,60,0,1,1,0,aged-out


In [15]:
_m.columns.tolist()

['Unnamed: 1',
 'timestamp',
 'session_id',
 'port_number',
 'application',
 'action',
 'matching_ip',
 'matched_column',
 'matched_ip',
 'total_bytes',
 'sent_bytes',
 'received_bytes',
 'total_packets',
 'packets_sent',
 'packets_received',
 'reason_for_session_end']

## Load all files

Loops through every hourly file and stacks matching rows only. Each row gets `source_hour` (a real `Timestamp`) and `source_file`.

In [16]:
match_frames = []

for i, path in enumerate(files, 1):
    hour = hour_from_filename(path)
    fname = os.path.basename(path)
    m = parse_file(path)
    if not m.empty:
        m.insert(0, 'source_hour', hour)
        m.insert(1, 'source_file', fname)
        match_frames.append(m)
    if i % 25 == 0 or i == len(files):
        print(f'  parsed {i}/{len(files)}')

matches_df = pd.concat(match_frames, ignore_index=True) if match_frames else pd.DataFrame()

print()
print(f'matches_df: {len(matches_df):,} rows  x  {matches_df.shape[1]} cols')

  parsed 25/190
  parsed 50/190
  parsed 75/190
  parsed 100/190
  parsed 125/190
  parsed 150/190
  parsed 175/190
  parsed 190/190

matches_df: 27,748 rows  x  18 cols


## Light cleanup

- If `read_fwf` split the session date into a leading `Unnamed:` column, combine it with the time column before parsing.
- Convert numeric columns in `matches_df` to real numbers.
- Parse the session `timestamp` into a real datetime.
- Strip whitespace from text columns.

In [17]:
# Session datetime: date often lands in the first unnamed column after read_fwf
unnamed_date_cols = [c for c in matches_df.columns if str(c).lower().startswith('unnamed')]
if unnamed_date_cols and 'timestamp' in matches_df.columns:
    date_col = unnamed_date_cols[0]
    combined = (
        matches_df[date_col].astype(str).str.strip()
        + ' '
        + matches_df['timestamp'].astype(str).str.strip()
    )
    matches_df['timestamp'] = pd.to_datetime(combined, format='%Y/%m/%d %H:%M:%S', errors='coerce')
    matches_df = matches_df.drop(columns=[date_col])
elif 'timestamp' in matches_df.columns:
    matches_df['timestamp'] = pd.to_datetime(
        matches_df['timestamp'], format='%Y/%m/%d %H:%M:%S', errors='coerce'
    )

numeric_cols = [
    'session_id', 'port_number',
    'total_bytes', 'sent_bytes', 'received_bytes',
    'total_packets', 'packets_sent', 'packets_received',
]
for col in numeric_cols:
    if col in matches_df.columns:
        matches_df[col] = pd.to_numeric(matches_df[col], errors='coerce')

for col in matches_df.select_dtypes(include='object').columns:
    matches_df[col] = matches_df[col].astype(str).str.strip()

matches_df.dtypes

source_hour               datetime64[us]
source_file                       object
timestamp                 datetime64[ns]
session_id                         int64
port_number                        int64
application                       object
action                            object
matching_ip                       object
matched_column                    object
matched_ip                        object
total_bytes                        int64
sent_bytes                         int64
received_bytes                     int64
total_packets                      int64
packets_sent                       int64
packets_received                   int64
reason_for_session_end            object
dtype: object

## Enrich with ThreatConnect address index

Loads `Splunk_TC_Address_Index.csv` the same way as `Splunk_TC_Address_Index.ipynb`, then left-joins each PA log match to ThreatConnect on **`matched_ip` = `indicator`** (IP address). Output: **`matches_tc_df`** — all session rows with TC metadata where the address exists in the index.

In [26]:
csv_path = r'H:\HTOC\notebooks\Gap Observation 2.0\Splunk_TC_Address_Index.csv'
results_df = pd.read_csv(csv_path)

left = matches_df.assign(
    join_ip=matches_df['matched_ip'].astype(str).str.strip()
)
right = results_df.assign(
    join_ip=results_df['indicator'].astype(str).str.strip()
)

matches_tc_df = left.merge(right, on='join_ip', how='left', suffixes=('', '_tc'))
matches_tc_df = matches_tc_df.drop(columns=['join_ip'])

in_index = matches_tc_df['indicator'].notna()
print(f'matches rows: {len(matches_tc_df):,}')
print(f'with TC index hit (matched_ip in indicator): {in_index.sum():,} ({100 * in_index.mean():.1f}%)')
print(f'no TC row for IP: {(~in_index).sum():,}')
matches_tc_df.loc[in_index].head(10)

matches rows: 27,748
with TC index hit (matched_ip in indicator): 27,748 (100.0%)
no TC row for IP: 0


,source_hour,source_file,timestamp,session_id,port_number,application,action,matching_ip,matched_column,matched_ip,total_bytes,sent_bytes,received_bytes,total_packets,packets_sent,packets_received,reason_for_session_end,id,dateAdded,ownerId,ownerName,webLink,type,lastModified,rating,confidence,threatAssessRating,threatAssessConfidence,threatAssessScore,threatAssessScoreObserved,threatAssessScoreFalsePositive,calScore,summary,observations,lastObserved,falsePositives,falsePositiveReportedByUser,privateFlag,active,activeLocked,ip,legacyLink,tags.data,associatedGroups.data,description,source,lastFalsePositive,indicator
0,2026-05-05 10:00:00,pa_traffic_observations.log-2026050510,2026-05-05 09:13:29,6191117,47759,incomplete,allow,193.163.125.51,src_ip,193.163.125.51,62,62,0,1,1,0,aged-out,5629499542036642,2025-05-28T19:30:54Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indicators/56294...,Address,2026-05-06T10:15:44Z,3.0,34.0,1.00,40.00,312,167,0,180,193.163.125.51,6797000,2026-05-06T00:00:00Z,0,False,False,True,False,193.163.125.51,https://hvs.threatconnect.com/auth/indicators/details/ad...,"[{'id': 2076981, 'name': 'SOAR Indicator PB', 'lastUsed'...",NaN,RITM0588707 / TASK0886419,NaN,NaN,193.163.125.51
1,2026-05-05 10:00:00,pa_traffic_observations.log-2026050510,2026-05-05 09:13:29,1149322950,41415,ssdp,allow,172.202.118.67,src_ip,172.202.118.67,141,141,0,1,1,0,aged-out,5629499576019463,2025-11-03T15:45:20Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indicators/56294...,Address,2026-05-06T13:16:27Z,3.0,62.0,3.00,62.00,494,167,0,230,172.202.118.67,1749491,2026-05-06T00:00:00Z,0,False,False,True,False,172.202.118.67,https://hvs.threatconnect.com/auth/indicators/details/ad...,"[{'id': 2118242, 'name': 'Scanning CDN PB', 'lastUsed': ...",NaN,INC9282086,NaN,NaN,172.202.118.67
2,2026-05-05 10:00:00,pa_traffic_observations.log-2026050510,2026-05-05 09:13:29,1074243713,33108,incomplete,allow,193.163.125.52,src_ip,193.163.125.52,60,60,0,1,1,0,aged-out,5629499542036630,2025-05-28T19:30:47Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indicators/56294...,Address,2026-05-06T11:19:23Z,3.0,32.0,1.00,27.67,312,167,0,180,193.163.125.52,6322944,2026-05-06T00:00:00Z,0,False,False,True,False,193.163.125.52,https://hvs.threatconnect.com/auth/indicators/details/ad...,"[{'id': 2076981, 'name': 'SOAR Indicator PB', 'lastUsed'...",NaN,RITM0584685 / TASK0881439,NaN,NaN,193.163.125.52
3,2026-05-05 10:00:00,pa_traffic_observations.log-2026050510,2026-05-05 09:13:28,1110873964,37310,incomplete,allow,193.163.125.31,src_ip,193.163.125.31,60,60,0,1,1,0,aged-out,5629499542036634,2025-05-28T19:30:48Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indicators/56294...,Address,2026-05-01T07:07:07Z,3.0,33.0,1.00,28.00,312,167,0,180,193.163.125.31,6345789,2026-05-06T00:00:00Z,0,False,False,True,False,193.163.125.31,https://hvs.threatconnect.com/auth/indicators/details/ad...,"[{'id': 2076981, 'name': 'SOAR Indicator PB', 'lastUsed'...",NaN,RITM0584685 / TASK0881439,NaN,NaN,193.163.125.31
4,2026-05-05 10:00:00,pa_traffic_observations.log-2026050510,2026-05-05 09:13:28,1111004683,48793,incomplete,allow,91.191.209.46,src_ip,91.191.209.46,60,60,0,1,1,0,aged-out,6755399492290043,2026-02-19T17:06:58Z,1009000,Google Threat Intelligence,https://hvs.threatconnect.com/#/details/indicators/67553...,Address,2026-05-06T09:15:43Z,1.0,86.0,1.17,37.67,608,167,0,660,91.191.209.46,67557641,2026-05-06T00:00:00Z,0,False,False,True,False,91.191.209.46,https://hvs.threatconnect.com/auth/indicators/details/ad...,"[{'id': 1820247, 'name': 'malware', 'lastUsed': '2026-02...","[{'id': 5629499574003003, 'dateAdded': '2026-02-13T11:37...",This indicator did not match our detection criteria and ...,https://www.virustotal.com/gui/ip-address/91.191.209.46,NaN,91.191.209.46
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27743,202

## Order by `matched_ip`

Sorts **`matches_tc_df`** by `matched_ip` (trimmed string order) so session rows for the same address appear together. The full table below uses this ordering.

In [28]:
_sort = matches_tc_df['matched_ip'].astype(str).str.strip()
matches_tc_df = (
    matches_tc_df.assign(_sort_ip=_sort)
    .sort_values('_sort_ip', na_position='last')
    .drop(columns=['_sort_ip'])
    .reset_index(drop=True)
)
matches_tc_df

,source_hour,source_file,timestamp,session_id,port_number,application,action,matching_ip,matched_column,matched_ip,total_bytes,sent_bytes,received_bytes,total_packets,packets_sent,packets_received,reason_for_session_end,id,dateAdded,ownerId,ownerName,webLink,type,lastModified,rating,confidence,threatAssessRating,threatAssessConfidence,threatAssessScore,threatAssessScoreObserved,threatAssessScoreFalsePositive,calScore,summary,observations,lastObserved,falsePositives,falsePositiveReportedByUser,privateFlag,active,activeLocked,ip,legacyLink,tags.data,associatedGroups.data,description,source,lastFalsePositive,indicator
0,2026-05-06 17:00:00,pa_traffic_observations.log-2026050617,2026-05-06 16:15:09,1079479808,57780,incomplete,allow,103.203.59.0,src_ip,103.203.59.0,60,60,0,1,1,0,aged-out,6755399448005412,2025-05-19T11:57:22Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indicators/67553...,Address,2026-05-06T10:05:44Z,3.0,32.0,3.0,32.0,368,167,0,180,103.203.59.0,9074199,2026-05-06T00:00:00Z,0,False,False,True,False,103.203.59.0,https://hvs.threatconnect.com/auth/indicators/details/ad...,"[{'id': 2076981, 'name': 'SOAR Indicator PB', 'lastUsed'...",NaN,RITM0581780/TASK0877793,NaN,NaN,103.203.59.0
1,2026-05-09 02:00:00,pa_traffic_observations.log-2026050902,2026-05-09 01:13:31,72121745,59267,incomplete,allow,103.203.59.0,src_ip,103.203.59.0,60,60,0,1,1,0,aged-out,6755399448005412,2025-05-19T11:57:22Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indicators/67553...,Address,2026-05-06T10:05:44Z,3.0,32.0,3.0,32.0,368,167,0,180,103.203.59.0,9074199,2026-05-06T00:00:00Z,0,False,False,True,False,103.203.59.0,https://hvs.threatconnect.com/auth/indicators/details/ad...,"[{'id': 2076981, 'name': 'SOAR Indicator PB', 'lastUsed'...",NaN,RITM0581780/TASK0877793,NaN,NaN,103.203.59.0
2,2026-05-09 01:00:00,pa_traffic_observations.log-2026050901,2026-05-09 00:13:52,371828102,47726,incomplete,allow,103.203.59.0,src_ip,103.203.59.0,60,60,0,1,1,0,aged-out,6755399448005412,2025-05-19T11:57:22Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indicators/67553...,Address,2026-05-06T10:05:44Z,3.0,32.0,3.0,32.0,368,167,0,180,103.203.59.0,9074199,2026-05-06T00:00:00Z,0,False,False,True,False,103.203.59.0,https://hvs.threatconnect.com/auth/indicators/details/ad...,"[{'id': 2076981, 'name': 'SOAR Indicator PB', 'lastUsed'...",NaN,RITM0581780/TASK0877793,NaN,NaN,103.203.59.0
3,2026-05-12 06:00:00,pa_traffic_observations.log-2026051206,2026-05-12 05:13:06,1142217730,34096,incomplete,allow,103.203.59.0,src_ip,103.203.59.0,60,60,0,1,1,0,aged-out,6755399448005412,2025-05-19T11:57:22Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indicators/67553...,Address,2026-05-06T10:05:44Z,3.0,32.0,3.0,32.0,368,167,0,180,103.203.59.0,9074199,2026-05-06T00:00:00Z,0,False,False,True,False,103.203.59.0,https://hvs.threatconnect.com/auth/indicators/details/ad...,"[{'id': 2076981, 'name': 'SOAR Indicator PB', 'lastUsed'...",NaN,RITM0581780/TASK0877793,NaN,NaN,103.203.59.0
4,2026-05-08 17:00:00,pa_traffic_observations.log-2026050817,2026-05-08 16:12:55,1080324888,60588,incomplete,allow,103.203.59.0,src_ip,103.203.59.0,60,60,0,1,1,0,aged-out,6755399448005412,2025-05-19T11:57:22Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indicators/67553...,Address,2026-05-06T10:05:44Z,3.0,32.0,3.0,32.0,368,167,0,180,103.203.59.0,9074199,2026-05-06T00:00:00Z,0,False,False,True,False,103.203.59.0,https://hvs.threatconnect.com/auth/indicators/details/ad...,"[{'id': 2076981, 'name': 'SOAR Indicator PB', 'lastUsed'...",NaN,RITM0581780/TASK0877793,NaN,NaN,103.203.59.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27743,2026-05-12 07:00:00,pa_traffic_observations.log-2026051207,2026-05-12 06:13:00,1114388382,54245,incomplete,allow,91.196.152.92,src_ip,91.196.152.92,74,74,0,1,1,0,aged-out,6755399467420999,2025-08-13T15:08:40Z,9,HTOC Or

### All sessions (one row per session; ordered by `matched_ip`)

In [19]:
matches_tc_df

,source_hour,source_file,timestamp,session_id,port_number,application,action,matching_ip,matched_column,matched_ip,total_bytes,sent_bytes,received_bytes,total_packets,packets_sent,packets_received,reason_for_session_end
0,2026-05-05 10:00:00,pa_traffic_observations.log-2026050510,2026-05-05 09:13:29,6191117,47759,incomplete,allow,193.163.125.51,src_ip,193.163.125.51,62,62,0,1,1,0,aged-out
1,2026-05-05 10:00:00,pa_traffic_observations.log-2026050510,2026-05-05 09:13:29,1149322950,41415,ssdp,allow,172.202.118.67,src_ip,172.202.118.67,141,141,0,1,1,0,aged-out
2,2026-05-05 10:00:00,pa_traffic_observations.log-2026050510,2026-05-05 09:13:29,1074243713,33108,incomplete,allow,193.163.125.52,src_ip,193.163.125.52,60,60,0,1,1,0,aged-out
3,2026-05-05 10:00:00,pa_traffic_observations.log-2026050510,2026-05-05 09:13:28,1110873964,37310,incomplete,allow,193.163.125.31,src_ip,193.163.125.31,60,60,0,1,1,0,aged-out
4,2026-05-05 10:00:00,pa_traffic_observations.log-2026050510,2026-05-05 09:13:28,1111004683,48793,incomplete,allow,91.191.209.46,src_ip,91.191.209.46,60,60,0,1,1,0,aged-out
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27743,2026-05-13 07:00:00,pa_traffic_observations.log-2026051307,2026-05-13 06:11:40,346034526,55132,incomplete,allow,193.163.125.216,src_ip,193.163.125.216,60,60,0,1,1,0,aged-out
27744,2026-05-13 07:00:00,pa_traffic_observations.log-2026051307,2026-05-13 06:11:40,1117079757,42007,incomplete,allow,176.65.148.92,src_ip,176.65.148.92,60,60,0,1,1,0,aged-out
27745,2026-05-13 07:00:00,pa_traffic_observations.log-2026051307,2026-05-13 06:11:40,0,34283,not-applicable,drop,64.62.156.200,src_ip,64.62.156.200,0,0,0,1,1,0,policy-deny
27746,2026-05-13 07:00:00,pa_traffic_observations.log-2026051307,2026-05-13 06:11:40,5654430,45281,stun,allow,65.49.1.115,src_ip,65.49.1.115,66,66,0,1,1,0,aged-out


## Quick summaries

In [20]:
per_hour = (
    matches_df.groupby('source_hour').size()
    .rename('matching_rows')
    .to_frame()
)
per_hour.head(24)

,matching_rows
source_hour,
2026-05-05 10:00:00,48
2026-05-05 11:00:00,74
2026-05-05 12:00:00,109
2026-05-05 13:00:00,105
2026-05-05 14:00:00,60
2026-05-05 15:00:00,51
2026-05-05 16:00:00,147
2026-05-05 17:00:00,138
2026-05-05 18:00:00,163


In [21]:
if 'matched_ip' in matches_df.columns:
    top_matched = (
        matches_df['matched_ip'].value_counts().head(20)
        .rename('sessions')
    )
    display(top_matched)

matched_ip
91.191.209.46     2411
5.188.206.194      459
176.65.148.92      422
89.190.156.34      342
176.65.148.147     310
176.65.149.182     264
198.235.24.79      171
205.210.31.220     168
198.235.24.109     164
205.210.31.53      157
205.210.31.232     154
205.210.31.60      154
198.235.24.167     153
205.210.31.212     150
205.210.31.244     149
205.210.31.81      148
198.235.24.239     148
198.235.24.74      147
205.210.31.66      146
205.210.31.55      145
Name: sessions, dtype: int64

In [22]:
if {'application', 'action'}.issubset(matches_df.columns):
    app_breakdown = (
        matches_df.groupby(['application', 'action']).size()
        .sort_values(ascending=False).head(20)
        .rename('sessions')
    )
    display(app_breakdown)

application        action
incomplete         allow     23910
not-applicable     drop        886
insufficient-data  allow       728
dtls               allow       542
dns-base           allow       197
ntp-base           allow        91
ipsec-esp-udp      allow        85
ike                allow        79
ssdp               allow        75
unknown-udp        allow        61
stun               allow        59
netbios-ns         allow        58
tftp               allow        58
pcanywhere-base    allow        57
gtpv1-c            allow        54
sip                allow        45
kerberos           allow        45
rmcp               allow        41
l2tp               allow        41
xdmcp              allow        39
Name: sessions, dtype: int64